# Legal Judgment Information Extraction — Transformer NER

Portfolio version of the transformer-based Named Entity Recognition pipeline used in a legal-document information extraction project on Indonesian criminal court decisions.

The project evaluates domain-specific and general Indonesian transformer models for extracting structured legal information from court-decision text. The broader study used **250 criminal court decisions** and defined **13 legal entity types**.

This notebook preserves the transformer NER workflow while removing local paths, long training logs, raw document outputs, and experimental material that is not necessary for a public portfolio.

> **Important note:** The metrics shown by this notebook are kept as produced by the original code. The study paper reports summary metrics separately, and not every stored notebook output is numerically identical to the paper table.

## Entity Types

- NOMOR_PUTUSAN
- LOKASI_PENGADILAN
- NAMA_HAKIM
- TANGGAL_PUTUSAN
- NAMA_TERDAKWA
- TANGGAL_LAHIR_TERDAKWA
- AGAMA_TERDAKWA
- PASAL_DAKWAAN
- VONIS_PENJARA
- STATUS_PUTUSAN
- MODUS_TINDAK_PIDANA
- KORBAN
- AKIBAT_TINDAK_PIDANA

---


# 1. PRE-PROCESSING DATASET UNTUK NER (DATASET CONSTRUCTION).

In [ ]:
import json
import pandas as pd

# =============================
# 1. LOAD & MERGE ANNOTATIONS
# =============================
file_path = "path/to/annotated_legal_entities.json"

with open(file_path, "r", encoding="utf-8") as f:
    raw = json.load(f)

print("Total items loaded:", len(raw))


def extract_annotations(item):
    """Ambil anotasi dari annotations atau drafts."""
    results = []

    # Jika annotations ada
    if "annotations" in item and len(item["annotations"]) > 0:
        for r in item["annotations"][0].get("result", []):
            results.append(r)

    # Jika drafts berisi hasil labeling
    if "drafts" in item and len(item["drafts"]) > 0:
        for r in item["drafts"][0].get("result", []):
            results.append(r)

    return results


# =============================
# 2. CLEAN ANNOTATIONS
# =============================
def clean_entities(results):
    clean = []

    for r in results:
        if "value" not in r:
            continue
        v = r["value"]

        if "start" not in v or "end" not in v or "labels" not in v:
            continue

        start = v["start"]
        end = v["end"]

        # Skip range invalid
        if start >= end:
            continue

        label = v["labels"][0]

        # Skip label tidak dipakai
        if label == "NAMA_FILE":
            continue

        clean.append({
            "start": start,
            "end": end,
            "label": label
        })

    # Urutkan anotasi berdasarkan start
    clean.sort(key=lambda x: x["start"])
    return clean


# =============================
# 3. BUILD BIO FROM CHARACTER MASK
# =============================
def build_bio(text, entities):
    bio = ["O"] * len(text)

    for ent in entities:
        s, e, label = ent["start"], ent["end"], ent["label"]

        bio[s] = f"B-{label}"
        for i in range(s + 1, e):
            if i < len(bio):
                bio[i] = f"I-{label}"

    return bio


# =============================
# 4. CONVERT TO TOKEN + BIO BY WORD
# =============================
def tokenize_and_align(text, bio):
    tokens = []
    tags = []

    cursor = 0
    words = text.split()

    for word in words:
        idx = text.find(word, cursor)
        if idx == -1:
            cursor += len(word) + 1
            continue

        tokens.append(word)
        tags.append(bio[idx])  # label karakter pertama kata

        cursor = idx + len(word)

    return tokens, tags


# =============================
# 5. PROCESS ALL DOCUMENTS
# =============================
final = []

for item in raw:
    if "data" not in item or "text" not in item["data"]:
        continue

    text = item["data"]["text"]

    raw_results = extract_annotations(item)
    entities = clean_entities(raw_results)

    bio = build_bio(text, entities)

    tokens, tags = tokenize_and_align(text, bio)

    final.append({
        "tokens": tokens,
        "ner_tags": tags
    })

df = pd.DataFrame(final)

df.to_csv("ner_dataset_ready.csv", index=False)
print("Saved dataset to ner_dataset_ready.csv")
print("Total processed:", len(df))
print(df.head())


# 2. DATASET ANOTASI

## 2.1 DATASET NER (INDO-LEGAL-BERT)


In [ ]:
# ============================================================
# NER TRAINING (CLEAN, NO LEAKAGE, STRICT BIO)
# ============================================================

import json, os, random
from collections import Counter
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    BertForTokenClassification,
    BertConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)

# ---------------- USER CONFIG ----------------
DATA_JSON = "path/to/annotated_legal_entities.json"
MODEL_CHECKPOINT = "archi-ai/Indo-LegalBERT"
OUTPUT_DIR = "outputs/indolegalbert_ner"
MAX_LENGTH = 256
SEED = 42
FP16 = True
# ---------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# 1. LOAD & CLEAN DATA
# ============================================================

with open(DATA_JSON, "r", encoding="utf-8") as f:
    raw = json.load(f)

def extract_results(item):
    out = []
    if item.get("annotations"):
        out.extend(item["annotations"][0].get("result", []))
    if item.get("drafts"):
        out.extend(item["drafts"][0].get("result", []))
    return out

def clean_results(results):
    ents = []
    for r in results:
        v = r.get("value", {})
        if not all(k in v for k in ("start", "end", "labels")):
            continue
        if v["start"] >= v["end"]:
            continue
        label = v["labels"][0]
        if label == "NAMA_FILE":
            continue
        ents.append({
            "start": v["start"],
            "end": v["end"],
            "label": label
        })
    return sorted(ents, key=lambda x: x["start"])

processed = []
for idx, item in enumerate(raw):
    text = item.get("data", {}).get("text")
    if not text:
        continue

    entities = clean_results(extract_results(item))

    # char-level BIO
    char_tags = ["O"] * len(text)
    for ent in entities:
        char_tags[ent["start"]] = f"B-{ent['label']}"
        for i in range(ent["start"] + 1, min(ent["end"], len(text))):
            char_tags[i] = f"I-{ent['label']}"

    tokens, tags = [], []
    cursor = 0
    for word in text.split():
        pos = text.find(word, cursor)
        if pos == -1:
            continue
        tokens.append(word)
        tags.append(char_tags[pos])
        cursor = pos + len(word)

    if tokens:
        processed.append({
            "doc_id": idx,
            "tokens": tokens,
            "ner_tags": tags
        })

df = pd.DataFrame(processed)
print("Total documents:", len(df))

# ============================================================
# 2. LABEL SET (INCLUDE 'O')
# ============================================================

labels = set(["O"])
for tags in df["ner_tags"]:
    labels.update(tags)

label_list = sorted(labels)
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print("Num labels:", len(label_list))

# ============================================================
# 3. LEAKAGE-SAFE SPLIT (DOCUMENT LEVEL)
# ============================================================

train_ids = df.sample(frac=0.8, random_state=SEED)["doc_id"]
train_df = df[df.doc_id.isin(train_ids)]
test_df = df[~df.doc_id.isin(train_ids)]

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "test": Dataset.from_pandas(test_df)
})

# ============================================================
# 4. TOKENIZATION + STRICT BIO ALIGNMENT
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH
    )

    aligned_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            else:
                tag = labels[word_id]
                if tag == "O":
                    label_ids.append(label2id["O"])
                else:
                    if word_id != prev_word:
                        label_ids.append(label2id[tag])
                    else:
                        label_ids.append(label2id[tag.replace("B-", "I-")])
            prev_word = word_id

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

tokenized_ds = dataset.map(tokenize_and_align_labels, batched=True)

# ============================================================
# 5. MODEL
# ============================================================

config = BertConfig.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model = BertForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    config=config,
    ignore_mismatched_sizes=True
)

model.gradient_checkpointing_enable()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ============================================================
# 6. METRICS (OVERALL + PER LABEL)
# ============================================================

metric = evaluate.load("seqeval")

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    true_preds, true_labels = [], []
    for p_seq, l_seq in zip(preds, labels):
        tp, tl = [], []
        for p_i, l_i in zip(p_seq, l_seq):
            if l_i != -100:
                tp.append(id2label[p_i])
                tl.append(id2label[l_i])
        true_preds.append(tp)
        true_labels.append(tl)

    res = metric.compute(predictions=true_preds, references=true_labels)

    out = {
        "precision": res["overall_precision"],
        "recall": res["overall_recall"],
        "f1": res["overall_f1"]
    }

    for k, v in res.items():
        if isinstance(v, dict) and "f1" in v:
            out[f"f1_{k}"] = v["f1"]

    return out

# ============================================================
# 7. TRAINING
# ============================================================

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=15,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=FP16,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# ============================================================
# 8. TRAIN & SAVE
# ============================================================

trainer.train()

model.save_pretrained(OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Training complete. Model saved to:", OUTPUT_DIR)


## 6. IndoBERT-base (Transformer ringan)

In [ ]:
# ============================================================
# NER TRAINING - INDOBERT BASE (REALISTIC, NO LEAKAGE)
# ============================================================

import json, random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    BertForTokenClassification,
    BertConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)

# ================= CONFIG =================
DATA_JSON = "path/to/annotated_legal_entities.json"
MODEL_CHECKPOINT = "indobenchmark/indobert-base-p1"
OUTPUT_DIR = "indobert_ner_realistic"
MAX_LENGTH = 256
SEED = 42
# ==========================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# 1. LOAD & PREPARE DATA (DOCUMENT LEVEL)
# ============================================================

with open(DATA_JSON, encoding="utf-8") as f:
    raw = json.load(f)

def extract_entities(item):
    out = []
    if item.get("annotations"):
        out += item["annotations"][0]["result"]
    if item.get("drafts"):
        out += item["drafts"][0]["result"]
    return out

processed = []

for doc_id, item in enumerate(raw):
    text = item["data"]["text"]
    entities = extract_entities(item)

    char_tags = ["O"] * len(text)
    for r in entities:
        v = r["value"]
        label = v["labels"][0]
        s, e = v["start"], v["end"]
        char_tags[s] = f"B-{label}"
        for i in range(s + 1, e):
            char_tags[i] = f"I-{label}"

    tokens, tags = [], []
    cursor = 0
    for w in text.split():
        pos = text.find(w, cursor)
        if pos == -1:
            continue
        tokens.append(w)
        tags.append(char_tags[pos])
        cursor = pos + len(w)

    if tokens:
        processed.append({
            "doc_id": doc_id,
            "tokens": tokens,
            "ner_tags": tags
        })

df = pd.DataFrame(processed)

# ============================================================
# 2. LABEL SET
# ============================================================

labels = sorted({t for row in df.ner_tags for t in row})
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

# ============================================================
# 3. DOCUMENT-LEVEL SPLIT (ANTI LEAKAGE)
# ============================================================

train_ids = df.sample(frac=0.75, random_state=SEED).doc_id
train_df = df[df.doc_id.isin(train_ids)]
test_df  = df[~df.doc_id.isin(train_ids)]

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "test": Dataset.from_pandas(test_df)
})

# ============================================================
# 4. TOKENIZATION & BIO ALIGNMENT
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_align(batch):
    tok = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH
    )

    labels_out = []
    for i, labs in enumerate(batch["ner_tags"]):
        word_ids = tok.word_ids(batch_index=i)
        prev = None
        ids = []
        for w in word_ids:
            if w is None:
                ids.append(-100)
            else:
                tag = labs[w]
                if w != prev:
                    ids.append(label2id[tag])
                else:
                    ids.append(label2id[tag.replace("B-", "I-")])
            prev = w
        labels_out.append(ids)

    tok["labels"] = labels_out
    return tok

dataset = dataset.map(tokenize_align, batched=True)

# ============================================================
# 5. MODEL
# ============================================================

config = BertConfig.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
)

model = BertForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    config=config
)

model.gradient_checkpointing_enable()

# ============================================================
# 6. METRICS
# ============================================================

metric = evaluate.load("seqeval")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    labels = p.label_ids

    true_preds, true_labels = [], []
    for pr, la in zip(preds, labels):
        tp, tl = [], []
        for p_i, l_i in zip(pr, la):
            if l_i != -100:
                tp.append(id2label[p_i])
                tl.append(id2label[l_i])
        true_preds.append(tp)
        true_labels.append(tl)

    r = metric.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": r["overall_precision"],
        "recall": r["overall_recall"],
        "f1": r["overall_f1"]
    }

# ============================================================
# 7. TRAINING (REGULARIZED)
# ============================================================

args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",          

    save_total_limit=1,             
    load_best_model_at_end=True,    
    metric_for_best_model="f1",     

    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=5,             #
    weight_decay=0.05,              # 🔥 REGULARISASI

    fp16=True,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ IndoBERT NER REALISTIC selesai")